# Model Name / Version Distribution
Monthly percent of booked loans by `risk_model_name` and `risk_model_version`,
split into two tables:
1. **KMX** (Mountain models)
2. **nonKMX** (Franchise models and any legacy names)

Data pulled from `edwnpi.los_deal_current_fact` from 2020-01-01 onward.

In [6]:
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
pd.set_option('display.max_rows', 200)

START_DATE = '2020-01-01'
END_DATE = None

run_every_query = True
PICKLE_PATH = '../../cache/model_mix_v1.pkl'


def run_sql(query, connection=None):
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    warnings.filterwarnings("ignore", category=UserWarning)
    df = pd.read_sql_query(sql=query, con=connection)
    warnings.filterwarnings("default", category=UserWarning)
    return df


def store_pickle(data, filename):
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

In [7]:
QUERY = f"""
SELECT
    cd.book_date,
    cd.risk_model_name,
    cd.risk_model_version,
    dru.riskdealergroup AS lob,
    cd.con_amount_financed_back AS amt_financed,
    cd.con_risk_model_score AS model_score,
    cd.account_number
FROM edwnpi.los_deal_current_fact AS cd
LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
    ON dru.dealer_number = cd.dealer_number
WHERE dru.riskdealergroup IN ('FRN','AN','STG','FLD','ENT','KMX')
  AND cd.data_source_name != 'SPARTAN'
  AND cd.con_amount_financed_back <= 100000
  AND cd.con_risk_model_score > 0
  AND cd.book_date IS NOT NULL
  AND cd.book_date >= '{START_DATE}'
"""

os.makedirs('cache', exist_ok=True)

if run_every_query or not os.path.exists(PICKLE_PATH):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        df = run_sql(QUERY, connection=conn)
    store_pickle(df, PICKLE_PATH)
    print('Query complete, cached to pickle')
else:
    df = get_pickle(PICKLE_PATH)
    print('Loaded from pickle')

df['book_date'] = pd.to_datetime(df['book_date'])
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()
df = df[df['book_date'] <= end_date].copy()

df['month'] = df['book_date'].dt.to_period('M')
df['risk_model_name'] = df['risk_model_name'].fillna('UNKNOWN').astype(str).str.strip()
df['risk_model_version'] = df['risk_model_version'].fillna(0)
df['model_label'] = df['risk_model_name'] + ' ' + df['risk_model_version'].astype(str)
df['is_kmx'] = df['lob'] == 'KMX'

df['month_str'] = df['month'].dt.year.astype(str) + ' M' + df['month'].dt.month.astype(str).str.zfill(2)

print(f"Total rows: {len(df):,}")
print(f"Date range: {df['book_date'].min().date()} to {df['book_date'].max().date()}")
print(f"\nDistinct model labels: {sorted(df['model_label'].unique())}")
print(f"\nKMX rows: {df['is_kmx'].sum():,}  |  nonKMX rows: {(~df['is_kmx']).sum():,}")

Query complete, cached to pickle
Total rows: 822,617
Date range: 2020-01-02 to 2026-08-13

Distinct model labels: ['FRANCHISE 3.0', 'FRANCHISE 3.1', 'Franchise 3.1', 'Franchise 4.0', 'Franchise 4.1', 'MOUNTAIN 1.0', 'MOUNTAIN 2.0', 'MOUNTAIN 2.1', 'MOUNTAIN 3.0', 'MOUNTAIN 3.1', 'Mountain 3.0', 'Mountain 3.1', 'Mountain 3.2', 'Mountain 4.1', 'Mountain 4.2', 'PANDA 1.0', 'PANDA 2.0', 'UNKNOWN 0.0']

KMX rows: 431,833  |  nonKMX rows: 390,784


In [8]:
# --- KMX Model Mix Table ---
kmx = df[df['is_kmx']].copy()

kmx_counts = kmx.groupby(['month_str', 'model_label']).size().unstack(fill_value=0)
kmx_totals = kmx_counts.sum(axis=1)
kmx_pct = kmx_counts.div(kmx_totals, axis=0) * 100
kmx_pct['total_loans'] = kmx_totals

kmx_pct = kmx_pct.round(1)
kmx_pct['total_loans'] = kmx_pct['total_loans'].astype(int)

print('KMX Model Mix (% of monthly loans)')
print('=' * 80)
display(kmx_pct)

# --- KMX Weighted Average Model Score per model_label ---
kmx_wtd_ms = kmx.groupby(['month_str', 'model_label']).apply(
    lambda g: np.average(g['model_score'], weights=g['amt_financed']),
    include_groups=False,
).unstack()

kmx_all_ms = kmx.groupby('month_str').apply(
    lambda g: np.average(g['model_score'], weights=g['amt_financed']),
    include_groups=False,
)
kmx_wtd_ms['All KMX'] = kmx_all_ms
kmx_wtd_ms = kmx_wtd_ms.round(1)

print('\nKMX Weighted Avg Model Score (by amt_financed)')
print('=' * 80)
kmx_wtd_ms

KMX Model Mix (% of monthly loans)


model_label,FRANCHISE 3.0,MOUNTAIN 1.0,MOUNTAIN 2.0,MOUNTAIN 2.1,MOUNTAIN 3.0,MOUNTAIN 3.1,Mountain 3.0,Mountain 3.1,Mountain 3.2,Mountain 4.1,Mountain 4.2,total_loans
month_str,,,,,,,,,,,,
2020 M01,0.0,35.0,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5661
2020 M02,0.0,0.2,99.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7079
2020 M03,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9749
2020 M04,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3633
2020 M05,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6444
2020 M06,0.0,0.0,96.7,3.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7816
2020 M07,0.0,0.0,90.3,9.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8559
2020 M08,0.0,0.0,79.5,20.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7406
2020 M09,0.0,0.0,50.6,49.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6565



KMX Weighted Avg Model Score (by amt_financed)


model_label,FRANCHISE 3.0,MOUNTAIN 1.0,MOUNTAIN 2.0,MOUNTAIN 2.1,MOUNTAIN 3.0,MOUNTAIN 3.1,Mountain 3.0,Mountain 3.1,Mountain 3.2,Mountain 4.1,Mountain 4.2,All KMX
month_str,,,,,,,,,,,,
2020 M01,NaN,135.2,136.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.1
2020 M02,NaN,137.1,136.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.1
2020 M03,NaN,NaN,135.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,135.6
2020 M04,NaN,NaN,136.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.5
2020 M05,NaN,NaN,138.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.4
2020 M06,NaN,NaN,139.2,139.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.2
2020 M07,NaN,NaN,139.4,138.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.3
2020 M08,NaN,NaN,138.8,138.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.6
2020 M09,NaN,NaN,138.3,138.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.3


In [9]:
# --- NonKMX Model Mix Table ---
nonkmx = df[~df['is_kmx']].copy()

nonkmx_counts = nonkmx.groupby(['month_str', 'model_label']).size().unstack(fill_value=0)
nonkmx_totals = nonkmx_counts.sum(axis=1)
nonkmx_pct = nonkmx_counts.div(nonkmx_totals, axis=0) * 100
nonkmx_pct['total_loans'] = nonkmx_totals

nonkmx_pct = nonkmx_pct.round(1)
nonkmx_pct['total_loans'] = nonkmx_pct['total_loans'].astype(int)

print('NonKMX Model Mix (% of monthly loans)')
print('=' * 80)
display(nonkmx_pct)

# --- NonKMX Weighted Average Model Score per model_label ---
nonkmx_wtd_ms = nonkmx.groupby(['month_str', 'model_label']).apply(
    lambda g: np.average(g['model_score'], weights=g['amt_financed']),
    include_groups=False,
).unstack()

nonkmx_all_ms = nonkmx.groupby('month_str').apply(
    lambda g: np.average(g['model_score'], weights=g['amt_financed']),
    include_groups=False,
)
nonkmx_wtd_ms['All nonKMX'] = nonkmx_all_ms
nonkmx_wtd_ms = nonkmx_wtd_ms.round(1)

print('\nNonKMX Weighted Avg Model Score (by amt_financed)')
print('=' * 80)
nonkmx_wtd_ms

NonKMX Model Mix (% of monthly loans)


model_label,FRANCHISE 3.0,FRANCHISE 3.1,Franchise 3.1,Franchise 4.0,Franchise 4.1,PANDA 1.0,PANDA 2.0,UNKNOWN 0.0,total_loans
month_str,,,,,,,,,
2020 M01,0.0,0.0,0.0,0.0,0.0,70.3,29.7,0.0,3729
2020 M02,0.0,0.0,0.0,0.0,0.0,63.8,36.2,0.0,3683
2020 M03,0.0,0.0,0.0,0.0,0.0,59.0,41.0,0.0,6237
2020 M04,0.0,0.0,0.0,0.0,0.0,57.2,42.8,0.0,3702
2020 M05,0.0,0.0,0.0,0.0,0.0,24.6,75.4,0.0,4389
2020 M06,0.0,0.0,0.0,0.0,0.0,16.3,83.7,0.0,5041
2020 M07,0.0,0.0,0.0,0.0,0.0,0.8,99.2,0.0,4984
2020 M08,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,4242
2020 M09,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,3688



NonKMX Weighted Avg Model Score (by amt_financed)


model_label,FRANCHISE 3.0,FRANCHISE 3.1,Franchise 3.1,Franchise 4.0,Franchise 4.1,PANDA 1.0,PANDA 2.0,UNKNOWN 0.0,All nonKMX
month_str,,,,,,,,,
2020 M01,NaN,NaN,NaN,NaN,NaN,129.9,132.1,NaN,130.6
2020 M02,NaN,NaN,NaN,NaN,NaN,131.6,131.9,NaN,131.7
2020 M03,NaN,NaN,NaN,NaN,NaN,131.2,131.5,NaN,131.3
2020 M04,NaN,NaN,NaN,NaN,NaN,130.8,131.4,NaN,131.0
2020 M05,NaN,NaN,NaN,NaN,NaN,132.8,134.1,NaN,133.8
2020 M06,NaN,NaN,NaN,NaN,NaN,134.0,135.3,NaN,135.1
2020 M07,NaN,NaN,NaN,NaN,NaN,134.7,135.6,NaN,135.6
2020 M08,NaN,NaN,NaN,NaN,NaN,130.0,135.7,NaN,135.7
2020 M09,NaN,NaN,NaN,NaN,NaN,NaN,135.5,NaN,135.5


In [10]:
# --- Combined (KMX + nonKMX) Model Mix Table ---
combined_counts = df.groupby(['month_str', 'model_label']).size().unstack(fill_value=0)
combined_totals = combined_counts.sum(axis=1)
combined_pct = combined_counts.div(combined_totals, axis=0) * 100
combined_pct['total_loans'] = combined_totals

combined_pct = combined_pct.round(1)
combined_pct['total_loans'] = combined_pct['total_loans'].astype(int)

print('Combined (KMX + nonKMX) Model Mix (% of monthly loans)')
print('=' * 80)
combined_pct

Combined (KMX + nonKMX) Model Mix (% of monthly loans)


model_label,FRANCHISE 3.0,FRANCHISE 3.1,Franchise 3.1,Franchise 4.0,Franchise 4.1,MOUNTAIN 1.0,MOUNTAIN 2.0,MOUNTAIN 2.1,MOUNTAIN 3.0,MOUNTAIN 3.1,Mountain 3.0,Mountain 3.1,Mountain 3.2,Mountain 4.1,Mountain 4.2,PANDA 1.0,PANDA 2.0,UNKNOWN 0.0,total_loans
month_str,,,,,,,,,,,,,,,,,,,
2020 M01,0.0,0.0,0.0,0.0,0.0,21.1,39.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.9,11.8,0.0,9390
2020 M02,0.0,0.0,0.0,0.0,0.0,0.1,65.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.8,12.4,0.0,10762
2020 M03,0.0,0.0,0.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0,16.0,0.0,15986
2020 M04,0.0,0.0,0.0,0.0,0.0,0.0,49.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.9,21.6,0.0,7335
2020 M05,0.0,0.0,0.0,0.0,0.0,0.0,59.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,30.5,0.0,10833
2020 M06,0.0,0.0,0.0,0.0,0.0,0.0,58.8,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.4,32.8,0.0,12857
2020 M07,0.0,0.0,0.0,0.0,0.0,0.0,57.1,6.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,36.5,0.0,13543
2020 M08,0.0,0.0,0.0,0.0,0.0,0.0,50.5,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.4,0.0,11648
2020 M09,0.0,0.0,0.0,0.0,0.0,0.0,32.4,31.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.0,0.0,10253


In [11]:
EXCEL_OUTPUT = '../output/model_mix_distribution.xlsx'

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    kmx_pct.to_excel(writer, sheet_name='KMX Pct')
    kmx_wtd_ms.to_excel(writer, sheet_name='KMX Avg Score')
    nonkmx_pct.to_excel(writer, sheet_name='NonKMX Pct')
    nonkmx_wtd_ms.to_excel(writer, sheet_name='NonKMX Avg Score')
    combined_pct.to_excel(writer, sheet_name='Combined Pct')

print(f'Exported to {EXCEL_OUTPUT}')

Exported to ../output/model_mix_distribution.xlsx
